In [1]:
# ── Step 1: RHF + CCSD on H₂ ──────────────────────────────────────────────
# PySCF gives us: MO coefficients, one-body integrals, t1/t2 amplitudes,
# and FCI as a reference.

import numpy as np
from pyscf import gto, scf, cc

mol = gto.M(atom="H 0 0 0; H 0 0 2", basis="sto6g", unit='B', verbose=0)

mf = scf.RHF(mol)
mf.kernel()
print(f"RHF  energy: {mf.e_tot:.10f} Ha")

mycc = cc.CCSD(mf)
mycc.kernel()
print(f"CCSD energy: {mycc.e_tot:.10f} Ha")

RHF  energy: -1.0564298822 Ha
CCSD energy: -1.0960712831 Ha


In [2]:
# ── Step 2: configure JAX and define orbital dimensions ───────────────────

# import sys, os
# sys.path.insert(0, os.path.join("..", "ad_afqmc_prototype"))

from ad_afqmc_prototype import config
config.configure_once()   # enable float64, etc.

import jax
import jax.numpy as jnp

mo   = mf.mo_coeff          # (nao, norb)  MO coefficient matrix
norb = mo.shape[1]           # number of MOs
nocc = mol.nelectron // 2   # occupied MOs (closed-shell)
nvir = norb - nocc
print(f"norb={norb}  nocc={nocc}  nvir={nvir}")

norb=2  nocc=1  nvir=1


In [3]:
from ad_afqmc_prototype.staging import TrialInput

def _stage_pt2ccsd_input(obj):
    if obj.kind != "ccsd":
        raise ValueError(f"Unreachable: '{obj.kind}'.")
    
    t1 = obj.t1
    t2 = obj.t2

    t1 = np.asarray(t1)
    t2 = np.asarray(t2)
    t2 = t2.transpose(0, 2, 1, 3) # (i,j,a,b) -> (i,a,j,b)

    def _thouless(init_slater, t1):
        # Thouless transformation: |psi'> = exp(t1_ia a+ i)|psi>
        # init slater: mo_coeff of psi (in mo basis)
        # return mo_coeff of psi' (in mo basis)
        norb, nvir = t1.shape
        norb = nocc + nvir
        exp_t1 = np.eye(norb, dtype=np.float64)
        exp_t1[:nocc, nocc:] = t1
        # exp_t1 = jsp.linalg.expm(t1_full)
        return exp_t1.T @ init_slater
    
    norb, nvir = t1.shape
    norb = nocc + nvir
    mo_coeff = np.eye(norb, dtype=jnp.float64)[:,:nocc]
    mo_t = _thouless(mo_coeff, t1)

    data = {"mo": mo_coeff, "mo_t": mo_t, "t2": t2}
    return TrialInput(
        kind="pt2ccsd",
        data=data,
        norb_frozen=obj.norb_frozen,
        source_kind=obj.source,
    )

In [4]:
# my step 3 & 4 build the Hamiltonian and the trial
# just use what is availble in the code, don't do unnecessary tensor contractions

from ad_afqmc_prototype.staging import _stage_ham_input
from ad_afqmc_prototype.staging import StagedMfOrCc
from ad_afqmc_prototype.ham.chol import HamChol

obj = StagedMfOrCc(mycc, norb_frozen=None)
staged_ham = _stage_ham_input(obj, chol_cut=1e-5, verbose=True)

ham_data = HamChol(
    h0=jnp.array(staged_ham.h0),
    h1=jnp.array(staged_ham.h1),
    chol=jnp.array(staged_ham.chol),
    basis=staged_ham.basis,
)

# Generating Cholesky decomposition of ERIs.
# max number of cholesky vectors = 20
# iteration     0: delta_max = 0.774999
# iteration     1: delta_max = 5.02620587e-01: time = 1.02448463e-03
# iteration     2: delta_max = 5.15162271e-03: time = 8.85725021e-04
# iteration     3: delta_max = 1.11022302e-16: time = 1.80006027e-03
[stage] AO cholesky: nchol=3 in 0.02s


In [5]:
from dataclasses import dataclass
from jax import tree_util

@tree_util.register_pytree_node_class
@dataclass(frozen=True)
class Pt2ccsdTrial:
    """
    Restricted pt2CCSD trial in an MO basis where the reference
    determinant occupies the first nocc orbitals.

    Arrays:
      mo_coeff: (nocc, norb)               mo_coeff of |psi_0> in mo basis
      mo_t: (nocc, norb)                   mo_coeff |psi'> = e^t1 |psi_0> by Thouless Theorem in mo basis
      t2: (nocc, nvir, nocc, nvir)         doubles amplitudess t2_{i a j b} in mo basis
    """

    # t1: jax.Array
    mo_coeff: jax.Array  # (norb, nocc)
    mo_t: jax.Array # (norb, nocc)
    t2: jax.Array # (norb, nvir, norb, nvir)

    @property
    def nocc(self) -> int:
        return int(self.t2.shape[0])

    @property
    def nvir(self) -> int:
        return int(self.t2.shape[1])

    @property
    def norb(self) -> int:
        return int(self.nocc + self.nvir)

    def tree_flatten(self):
        children = (self.mo_coeff, self.mo_t, self.t2)
        aux = None
        return children, aux

    @classmethod
    def tree_unflatten(cls, aux, children):
        mo_coeff, mo_t, t2 = children
        return cls(
            mo_coeff = mo_coeff,
            mo_t = mo_t,
            t2 = t2,
        )

In [6]:
@dataclass(frozen=True)
class Pt2ccsdMeasCfg:
    memory_mode: str = "low"  # or Literal["low","high"]
    mixed_real_dtype: jnp.dtype = jnp.float64
    mixed_complex_dtype: jnp.dtype = jnp.complex128
    mixed_real_dtype_testing: jnp.dtype = jnp.float32
    mixed_complex_dtype_testing: jnp.dtype = jnp.complex64

@tree_util.register_pytree_node_class
@dataclass(frozen=True)
class Pt2ccsdMeasCtx:
    # half-rotated:
    rot_h1: jax.Array  # (nocc, norb)
    rot_chol: jax.Array  # (n_chol, nocc, norb)
    rot_chol_flat: jax.Array  # (n_chol, nocc*norb)
    cfg: Pt2ccsdMeasCfg  # static

    def tree_flatten(self):
        children = (self.rot_h1, self.rot_chol, self.rot_chol_flat)
        aux = (self.cfg,)
        return children, aux

    @classmethod
    def tree_unflatten(cls, aux, children):
        (cfg,) = aux
        rot_h1, rot_chol, rot_chol_flat = children
        return cls(rot_h1=rot_h1, rot_chol=rot_chol, rot_chol_flat=rot_chol_flat, cfg=cfg)

def build_meas_ctx(
    ham_data: HamChol, trial_data: Pt2ccsdTrial, cfg: Pt2ccsdMeasCfg = Pt2ccsdMeasCfg()
) -> Pt2ccsdMeasCtx:
    if ham_data.basis != "restricted":
        raise ValueError("pt2CCSD MeasOps currently assumes HamChol.basis == 'restricted'.")

    # chol = ham_data.chol  # (n_chol, norb, norb)
    # nocc = trial_data.nocc
    # rot_chol = chol[:, :nocc, :]  # (n_chol, nocc, norb)
    cH = trial_data.mo_coeff.conj().T  # (nocc, norb)
    rot_h1 = cH @ ham_data.h1  # (nocc, norb)
    rot_chol = jnp.einsum("pi,gij->gpj", cH, ham_data.chol, optimize="optimal")
    rot_chol_flat = rot_chol.reshape(rot_chol.shape[0], -1)

    return Pt2ccsdMeasCtx(rot_h1=rot_h1, rot_chol=rot_chol, rot_chol_flat=rot_chol_flat, cfg=cfg)

In [7]:
staged_trial = _stage_pt2ccsd_input(obj)

In [8]:
staged_trial = _stage_pt2ccsd_input(obj)

trial_data = Pt2ccsdTrial(
    mo_coeff=jnp.array(staged_trial.data['mo'][:,:nocc]),
    mo_t = staged_trial.data["mo_t"],
    t2 = staged_trial.data["t2"],
    )

In [9]:
# ── Step 5: build Pt2CCSDMeasCtx — precomputed measurement arrays ────────────
#
# Two arrays cached once to avoid recomputation inside the energy kernel:
#   rot_chol[g, p, q] = L^MO_{g, p<nocc, q}  (occupied-row slice of Cholesky)
#   lci1[g, p, i]     = sum_t L^MO_{g,pt} * ci1_{it}  (Cholesky × singles amplitudes)

# from ad_afqmc_prototype.meas.cisd import build_meas_ctx, CisdMeasCfg

cfg      = Pt2ccsdMeasCfg(memory_mode="low")  # vectorise over all Cholesky at once
meas_ctx = build_meas_ctx(ham_data, trial_data, cfg)

print(f"rot_chol shape: {meas_ctx.rot_chol.shape}   (nchol, nocc, norb)")

rot_chol shape: (3, 1, 2)   (nchol, nocc, norb)


In [ ]:
# ── Step 6: import the energy and overlap kernels ─────────────────────────

# from ad_afqmc_prototype.meas.cisd import energy_kernel_rw_rh
# from ad_afqmc_prototype.trial.cisd import overlap_r

In [10]:
from jax import lax
# from ad_afqmc_prototype.trial.rhf import overlap_r

# @jax.jit
def _greens_restricted(walker: jax.Array, mo_t: jax.Array) -> jax.Array:
    return (walker @ (jnp.linalg.inv(mo_t.T @ walker)) @ mo_t.T).T

# @jax.jit
def _greenp_from_green(green: jax.Array, nocc: int) -> jax.Array:
    norb = green.shape[0]
    return (green - jnp.eye(norb))[:,nocc:]

# @jax.jit
def energy_kernel_rw_rh(
    walker: jax.Array, ham_data: HamChol, meas_ctx: Pt2ccsdMeasCtx, trial_data: Pt2ccsdTrial
) -> jax.Array:
    mo_t, t2 = trial_data.mo_t, trial_data.t2
    nocc = trial_data.nocc

    green = _greens_restricted(walker, mo_t)  # (norb, norb)
    greenp = _greenp_from_green(green, nocc)  # (norb, nvir)

    # h0 = ham_data.h0
    h1 = ham_data.h1
    chol = ham_data.chol
    # rot_chol = meas_ctx.rot_chol

    hg = jnp.einsum("pq,pq->", h1, green, optimize="optimal")
    e1_0 = 2 * hg

    # double excitations
    t2g_c = jnp.einsum("iajb,ia->jb", t2, green[:nocc,nocc:], optimize="optimal")
    t2g_e = jnp.einsum("iajb,ib->ja", t2, green[:nocc,nocc:], optimize="optimal")
    t2_green_c = (greenp @ t2g_c.T) @ green[:nocc,:]
    t2_green_e = (greenp @ t2g_e.T) @ green[:nocc,:]
    t2_green = 2 * t2_green_c - t2_green_e
    t2g = 2 * t2g_c - t2g_e
    gt2g = jnp.einsum("ia,ia->", t2g, green[:nocc,nocc:], optimize="optimal")
    e1_2_1 = 2 * hg * gt2g
    e1_2_2 = -2 * jnp.einsum("pq,pq->", h1, t2_green, optimize="optimal")
    e1_2 = e1_2_1 + e1_2_2 # <exp(T1)HF|T2 h1|walker>/<exp(T1)HF|walker>

    # two body energy
    lg = jnp.einsum("gpq,pq->g", chol, green, optimize="optimal")

    # double excitations
    lt2g = jnp.einsum("gpq,pq->g", chol, t2_green, optimize="optimal")
    e2_2_2_1 = -lt2g @ lg

    def scanned_fun(carry, x):
        chol_i = x
        # e2_0
        gl_i = jnp.einsum("pr,qr->pq", green, chol_i, optimize="optimal")
        e2_0_1_i = (2*jnp.trace(gl_i))**2 / 2.0
        e2_0_2_i = - jnp.einsum('pq,qp->',gl_i,gl_i, optimize="optimal")
        carry[0] += e2_0_1_i + e2_0_2_i
        # e2_2_2_2
        lt2_green_i = jnp.einsum("pr,qr->pq", chol_i, t2_green, optimize="optimal")
        carry[1] += 0.5 * jnp.einsum("pq,pq->", gl_i, lt2_green_i, optimize="optimal")
        # e2_2_3
        glgp_i = jnp.einsum("iq,qa->ia", gl_i[:nocc,:], greenp, optimize="optimal")
        l2t2_1 = jnp.einsum("ia,jb,iajb->", glgp_i, glgp_i, t2, optimize="optimal")
        l2t2_2 = jnp.einsum("ib,ja,iajb->", glgp_i, glgp_i, t2, optimize="optimal")
        carry[2] += 2 * l2t2_1 - l2t2_2
        return carry, 0.0

    [e2_0, e2_2_2_2, e2_2_3], _ = lax.scan(scanned_fun, [0.0,0.0,0.0], chol)
    e2_2_1 = e2_0 * gt2g
    e2_2_2 = 4 * (e2_2_2_1 + e2_2_2_2)
    e2_2 = e2_2_1 + e2_2_2 + e2_2_3

    # <HF|walker>
    o0 = jnp.linalg.det(walker[:nocc,:nocc]) ** 2
    t1 = jnp.linalg.det(mo_t.T.conj() @ walker)**2 / o0 # <exp(T1)HF|walker>/<HF|walker>
    t2 = gt2g * t1 # <exp(T1)HF|T2|walker>/<HF|walker>
    e0 = (e1_0 + e2_0) * t1 # <exp(T1)HF|h1+h2|walker>/<HF|walker>
    e1 = (e1_2 + e2_2) * t1 # <exp(T1)HF|T2 (h1+h2)|walker>/<HF|walker>
    # ecisd = h0 + (e1_0 + e2_0 + e1_2 + e2_2) / (1 + gt2g)
    return t1, t2, e0, e1


In [21]:
def random_walker(walker):
    # k1, k2 = jax.random.split(key)
    # real and imaginary parts drawn from N(0,1)
    re = np.random.rand(*walker.shape)
    im = np.random.rand(*walker.shape)
    w  = (re + 1j * im) / 2
    return jnp.array(w)

In [53]:
walker = jnp.zeros((norb, nocc), dtype=complex)
walker = walker.at[:nocc, :].set(jnp.eye(nocc))
walker = random_walker(walker)
t1, t2, e0, e1 = energy_kernel_rw_rh(walker, ham_data, meas_ctx, trial_data)
ecisd = ham_data.h0 + (e0 + e1) / (t1 + t2)
print(ecisd, (t1 + t2))

(-1.0960712830066368-4.759849044824961e-11j) (1.0623596261189334-0.07009847725097543j)


In [17]:
from ad_afqmc_prototype.core.system import System
from ad_afqmc_prototype.trial.rhf import make_rhf_trial_ops
from ad_afqmc_prototype.meas.rhf  import make_rhf_meas_ops
from ad_afqmc_prototype.prop.afqmc import make_prop_ops, init_prop_state
from ad_afqmc_prototype.prop.types import QmcParams
from ad_afqmc_prototype.prop.blocks import block as qmc_block

sys   = System(norb=norb, nelec=(nocc, nocc), walker_kind="restricted")
trial_ops = make_rhf_trial_ops(sys)
meas_ops  = make_rhf_meas_ops(sys)
prop_ops  = make_prop_ops(ham_data.basis, sys.walker_kind)

params = QmcParams(
    dt            = 0.005,   # imaginary-time step
    n_walkers     = 200,      # walker population
    n_prop_steps  = 50,      # propagation steps per block
    n_blocks      = 200,     # sampling blocks
    n_eql_blocks  = 100,      # equilibration blocks (discarded)
    seed          = 17,
)

print(f"dt={params.dt}  n_walkers={params.n_walkers}  n_prop_steps={params.n_prop_steps}")
print(f"equlibrium imaginary time: {params.n_eql_blocks * params.n_prop_steps * params.dt:.2f} a.u.")
print(f"sampling imaginary time: {params.n_blocks * params.n_prop_steps * params.dt:.2f} a.u.")

dt=0.005  n_walkers=200  n_prop_steps=50
equlibrium imaginary time: 25.00 a.u.
sampling imaginary time: 50.00 a.u.


In [18]:
state = init_prop_state(
    sys=sys,
    ham_data=ham_data,
    trial_ops=trial_ops,
    trial_data=trial_data,
    meas_ops=meas_ops,
    params=params,
)

print(f"Initial walker batch shape: {state.walkers.shape}")
print(f"Initial weights (first 5):  {state.weights[:5]}")
print(f"Initial e_estimate: {float(state.e_estimate):.8f} Ha")
print(f"PYSCF-HF Energy: {float(mf.e_tot):.8f} Ha")
print(f"Initial mean |overlap|: {float(jnp.mean(jnp.abs(state.overlaps))):.6f}")

Initial walker batch shape: (200, 2, 1)
Initial weights (first 5):  [1. 1. 1. 1. 1.]
Initial e_estimate: -1.05642988 Ha
PYSCF-HF Energy: -1.05642988 Ha
Initial mean |overlap|: 1.000000


In [13]:
from ad_afqmc_prototype.core.system import System
from ad_afqmc_prototype.core.ops import TrialOps
from ad_afqmc_prototype.trial.rhf import overlap_r, get_rdm1

def make_pt2ccsd_trial_ops(sys: System) -> TrialOps:
    if sys.nup != sys.ndn:
        raise ValueError("Restricted pt2CCSD trial requires nup == ndn.")
    if sys.walker_kind.lower() != "restricted":
        raise ValueError(
            f"pt2CCSD trial currently supports only restricted walkers, got: {sys.walker_kind}"
        )
    return TrialOps(overlap=overlap_r, get_rdm1=get_rdm1)

In [14]:
from ad_afqmc_prototype.core.ops import MeasOps
from ad_afqmc_prototype.meas.rhf import force_bias_kernel_rw_rh, rdm1_kernel_rw

k_force_bias = "force_bias"
k_energy = "energy"
o_rdm1 = "rdm1"

def make_pt2ccsd_meas_ops(
    sys: System,
    memory_mode: str = "low",
    mixed_precision: bool = False,
    testing: bool = False,
) -> MeasOps:
    if sys.walker_kind.lower() != "restricted":
        raise ValueError(
            f"pt2CCSD MeasOps currently supports only restricted walkers, got: {sys.walker_kind}"
        )

    cfg = Pt2ccsdMeasCfg(
        memory_mode=memory_mode,
        mixed_real_dtype=jnp.float32 if mixed_precision else jnp.float64,
        mixed_complex_dtype=jnp.complex64 if mixed_precision else jnp.complex128,
        mixed_real_dtype_testing=jnp.float64 if testing else jnp.float32,
        mixed_complex_dtype_testing=jnp.complex128 if testing else jnp.complex64,
    )

    return MeasOps(
        overlap=overlap_r,
        build_meas_ctx=lambda ham_data, trial_data: build_meas_ctx(ham_data, trial_data, cfg),
        kernels={k_force_bias: force_bias_kernel_rw_rh, k_energy: energy_kernel_rw_rh},
        observables={o_rdm1: rdm1_kernel_rw},
    )

In [ ]:
# trial_ops = make_pt2ccsd_trial_ops(sys)
# meas_ops  = make_pt2ccsd_meas_ops(sys, memory_mode="low")
# from ad_afqmc_prototype import walkers as wk

# e_kernel = meas_ops.require_kernel(k_energy)
# t1_sp, t2_sp, e0_sp, e1_sp = wk.vmap_chunked(e_kernel, n_chunks=params.n_chunks, in_axes=(0, None, None, None))(
#     state.walkers, ham_data, meas_ctx, trial_data
# )

In [ ]:
# ham_data.h0 + (e0_sp + e1_sp) / (t1_sp + t2_sp)

Array([-1.09607128+0.j, -1.09607128+0.j, -1.09607128+0.j, -1.09607128+0.j,
       -1.09607128+0.j, -1.09607128+0.j, -1.09607128+0.j, -1.09607128+0.j,
       -1.09607128+0.j, -1.09607128+0.j, -1.09607128+0.j, -1.09607128+0.j,
       -1.09607128+0.j, -1.09607128+0.j, -1.09607128+0.j, -1.09607128+0.j,
       -1.09607128+0.j, -1.09607128+0.j, -1.09607128+0.j, -1.09607128+0.j,
       -1.09607128+0.j, -1.09607128+0.j, -1.09607128+0.j, -1.09607128+0.j,
       -1.09607128+0.j, -1.09607128+0.j, -1.09607128+0.j, -1.09607128+0.j,
       -1.09607128+0.j, -1.09607128+0.j, -1.09607128+0.j, -1.09607128+0.j,
       -1.09607128+0.j, -1.09607128+0.j, -1.09607128+0.j, -1.09607128+0.j,
       -1.09607128+0.j, -1.09607128+0.j, -1.09607128+0.j, -1.09607128+0.j,
       -1.09607128+0.j, -1.09607128+0.j, -1.09607128+0.j, -1.09607128+0.j,
       -1.09607128+0.j, -1.09607128+0.j, -1.09607128+0.j, -1.09607128+0.j,
       -1.09607128+0.j, -1.09607128+0.j, -1.09607128+0.j, -1.09607128+0.j,
       -1.09607128+0.j, -

In [ ]:
from typing import Any, Callable, NamedTuple, Protocol

import jax
import jax.numpy as jnp
from jax import lax, tree_util

from ad_afqmc_prototype import walkers as wk
from ad_afqmc_prototype.core.levels import LevelPack
from ad_afqmc_prototype.core.ops import MeasOps, TrialOps, k_energy
from ad_afqmc_prototype.core.system import System
from ad_afqmc_prototype.walkers import SrFn
from ad_afqmc_prototype.prop.types import PropOps, PropState, QmcParams, QmcParamsFp
from ad_afqmc_prototype.prop.blocks import BlockObs

def block_pt2(
    state: PropState,
    *,
    params: QmcParams,
    ham_data: Any,
    trial_data: Any,
    meas_ops: MeasOps,
    meas_ctx: Any,
    obs: BlockObs,
) -> tuple[PropState, BlockObs]:
    
    """
    no propagation, just measurement
    propagation is done by HF block
    measure elements needed by pt2CCSD energy: 
        <exp(T1)> <T2> <exp(T1)H> <T2H>
    """

    e_kernel = meas_ops.require_kernel(k_energy)
    
    pt2ccsd_samples = wk.vmap_chunked(e_kernel, n_chunks=params.n_chunks, in_axes=(0, None, None, None))(
        state.walkers, ham_data, meas_ctx, trial_data
    )

    t1s, t2s, e0s, e1s = pt2ccsd_samples
    wts = state.weights

    wt = jnp.sum(wts)
    t1 = jnp.sum(wts * t1s) / wt
    t2 = jnp.sum(wts * t2s) / wt
    e0 = jnp.sum(wts * e0s) / wt
    e1 = jnp.sum(wts * e1s) / wt

    obs_pt2 = BlockObs(
        scalars={"energy": e_block, "weight": w_sum},
        observables=obs_samples,
    )
    return obs

In [19]:
from ad_afqmc_prototype.driver import run_qmc
run_qmc(
        sys=sys,
        params=params,
        ham_data=ham_data,
        trial_data=trial_data,
        meas_ops=meas_ops,
        trial_ops=trial_ops,
        prop_ops=prop_ops,
        block_fn=qmc_block,
        state=None,
        meas_ctx=meas_ctx,
        target_error=None,
        mesh=None,
        observable_names=(),
    )


Equilibration:

        block           E_blk             W        nodes      t[s]
[eql    0/100]   -1.0564298822  2.000000e+02           0       0.0


[eql   20/100]   -1.0922313954  2.001568e+02           0       1.2
[eql   40/100]   -1.0960047250  2.000169e+02           0       2.1
[eql   60/100]   -1.0921494467  1.999850e+02           0       2.4
[eql   80/100]   -1.0906510809  1.999956e+02           0       2.7
[eql  100/100]   -1.0896622800  1.999817e+02           0       3.0

Sampling:

        block           E_avg       E_err         E_block             W         nodes    dt[s/bl]     t[s]
[blk   20/200]   -1.0912183469   1.373e-03     -1.0912183469  1.999901e+02           0      0.163       3.3
[blk   40/200]   -1.0935370008   1.330e-03     -1.0958555787  1.999966e+02           0      0.015       3.6
[blk   60/200]   -1.0931479234   1.167e-03     -1.0923695949  1.999487e+02           0      0.014       3.8
[blk   80/200]   -1.0936076629   1.249e-03     -1.0949866682  2.000094e+02           0      0.013       4.1
[blk  100/200]   -1.0942052726   1.192e-03     -1.0965960284  1.999597e+02           0      0.015       4.4
[blk  

QmcResult(mean_energy=-1.093430676363761, stderr_energy=0.0011074724115687125, block_energies=Array([-1.05642988, -1.0649249 , -1.07155296, -1.07855578, -1.08583065,
       -1.10045408, -1.08665088, -1.09549689, -1.08560658, -1.09462042,
       -1.09380086, -1.09648975, -1.10101522, -1.10103759, -1.13285221,
       -1.09245941, -1.08855908, -1.08898825, -1.09149067, -1.0959852 ,
       -1.09817383, -1.09141418, -1.10836829, -1.08891315, -1.09390304,
       -1.08691737, -1.08771132, -1.09569174, -1.11202183, -1.10159166,
       -1.0896238 , -1.09245692, -1.08789465, -1.09330075, -1.11417213,
       -1.10230492, -1.10022162, -1.09169536, -1.09688689, -1.09239979,
       -1.09256091, -1.09836515, -1.09384515, -1.09171803, -1.09171   ,
       -1.09665713, -1.0985251 , -1.09203526, -1.08972604, -1.08701083,
       -1.08482148, -1.0915366 , -1.09594048, -1.09595159, -1.09194241,
       -1.09214212, -1.09177562, -1.09069372, -1.0894945 , -1.08725736,
       -1.09183123, -1.08571039, -1.084876

In [ ]:
from typing import Any

import jax
import jax.numpy as jnp
# from jax import lax, tree_util

# from ad_afqmc_prototype import walkers as wk
# from ad_afqmc_prototype.core.levels import LevelPack
from ad_afqmc_prototype.core.ops import MeasOps, TrialOps, k_energy
from ad_afqmc_prototype.core.system import System
from ad_afqmc_prototype.walkers import SrFn
from ad_afqmc_prototype.prop.types import PropOps, PropState, QmcParams
from ad_afqmc_prototype.prop.blocks import BlockFn
from ad_afqmc_prototype.stat_utils import blocking_analysis_ratio, jackknife_ratios, rebin_observable, reject_outliers
from ad_afqmc_prototype.walkers import stochastic_reconfiguration

# from jax import lax
from jax.sharding import Mesh, NamedSharding
from jax.sharding import PartitionSpec as P

from ad_afqmc_prototype.driver import QmcResult, make_run_blocks
from functools import partial
import time
print = partial(print, flush=True)

# def run_qmc(
    # *,
#     sys: System,
#     params: QmcParams,
#     ham_data: Any,
#     trial_data: Any,
#     meas_ops: MeasOps,
#     trial_ops: TrialOps,
#     prop_ops: PropOps,
#     block_fn: BlockFn,
#     state: PropState | None = None,
#     meas_ctx: Any | None = None,
#     target_error: float | None = None,
#     mesh: Mesh | None = None,
#     observable_names: tuple[str, ...] = (),
# ) -> QmcResult:
    # """
    # equilibration blocks then sampling blocks.

    # Returns:
    #   QmcResult with energy statistics plus block-level observable estimates.
    # """
    
mesh = None
observable_names = ()
target_error = None
block_fn = qmc_block
state = None
meas_ctx = None

for name in observable_names:
    meas_ops.require_observable(name)

# build ctx
prop_ctx = prop_ops.build_prop_ctx(ham_data, trial_ops.get_rdm1(trial_data), params)
if meas_ctx is None:
    meas_ctx = meas_ops.build_meas_ctx(ham_data, trial_data)
if state is None:
    state = prop_ops.init_prop_state(
        sys=sys,
        ham_data=ham_data,
        trial_ops=trial_ops,
        trial_data=trial_data,
        meas_ops=meas_ops,
        params=params,
        mesh=mesh,
    )

if mesh is None or mesh.size == 1:
    block_fn_sr = block_fn
else:
    data_sh = NamedSharding(mesh, P("data"))
    sr_sharded = partial(stochastic_reconfiguration, data_sharding=data_sh)
    block_fn_sr = partial(block_fn, sr_fn=sr_sharded)

run_blocks = make_run_blocks(
    block_fn=block_fn_sr,
    sys=sys,
    params=params,
    trial_ops=trial_ops,
    meas_ops=meas_ops,
    prop_ops=prop_ops,
    observable_names=observable_names,
)

t0 = time.perf_counter()
t_mark = t0

print_every = params.n_eql_blocks // 5 if params.n_eql_blocks >= 5 else 0
block_e_eq = []
block_w_eq = []
block_obs_eq = {name: [] for name in observable_names}
block_e_eq.append(state.e_estimate)
block_w_eq.append(jnp.sum(state.weights))
print("\nEquilibration:\n")
if print_every:
    print(
        f"{'':4s}"
        f"{'block':>9s}  "
        f"{'E_blk':>14s}  "
        f"{'W':>12s}   "
        f"{'nodes':>10s}  "
        f"{'t[s]':>8s}"
    )
print(
    f"[eql {0:4d}/{params.n_eql_blocks}]  "
    f"{float(state.e_estimate):14.10f}  "
    f"{float(jnp.sum(state.weights)):12.6e}  "
    f"{int(state.node_encounters):10d}  "
    f"{0.0:8.1f}"
)
chunk = print_every if print_every > 0 else 1
for start in range(0, params.n_eql_blocks, chunk):
    n = min(chunk, params.n_eql_blocks - start)
    state, scalars_chunk, obs_chunk = run_blocks(
        state,
        ham_data=ham_data,
        trial_data=trial_data,
        meas_ctx=meas_ctx,
        prop_ctx=prop_ctx,
        n_blocks=n,
    )
    block_e_eq.extend(scalars_chunk["energy"].tolist())
    block_w_eq.extend(scalars_chunk["weight"].tolist())
    for i, name in enumerate(observable_names):
        block_obs_eq[name].append(obs_chunk[i])
    e_chunk = scalars_chunk["energy"]
    w_chunk = scalars_chunk["weight"]
    w_chunk_avg = jnp.mean(w_chunk)
    e_chunk_avg = jnp.mean(e_chunk * w_chunk) / w_chunk_avg
    elapsed = time.perf_counter() - t0
    print(
        f"[eql {start + n:4d}/{params.n_eql_blocks}]  "
        f"{float(e_chunk_avg):14.10f}  "
        f"{float(w_chunk_avg):12.6e}  "
        f"{int(state.node_encounters):10d}  "
        f"{elapsed:8.1f}"
    )
block_e_eq = jnp.asarray(block_e_eq)
block_w_eq = jnp.asarray(block_w_eq)
block_obs_eq = {
    name: (jnp.concatenate(block_obs_eq[name], axis=0) if len(block_obs_eq[name]) > 0 else None)
    for name in observable_names
}

# sampling
print("\nSampling:\n")
if target_error is None:
    target_error = 0.0
print_every = params.n_blocks // 10 if params.n_blocks >= 10 else 0
block_e_s = []
block_w_s = []
block_obs_s = {name: [] for name in observable_names}
if print_every:
    print(
        f"{'':4s}{'block':>9s}  {'E_avg':>14s}  {'E_err':>10s}  {'E_block':>14s}  "
        f"{'W':>12s}    {'nodes':>10s}  {'dt[s/bl]':>10s}  {'t[s]':>7s}"
    )

chunk = print_every if print_every > 0 else 1
for start in range(0, params.n_blocks, chunk):
    n = min(chunk, params.n_blocks - start)
    state, scalars_chunk, obs_chunk = run_blocks(
        state,
        ham_data=ham_data,
        trial_data=trial_data,
        meas_ctx=meas_ctx,
        prop_ctx=prop_ctx,
        n_blocks=n,
    )
    e_chunk = scalars_chunk["energy"]
    w_chunk = scalars_chunk["weight"]
    block_e_s.extend(e_chunk.tolist())
    block_w_s.extend(w_chunk.tolist())
    for i, name in enumerate(observable_names):
        block_obs_s[name].append(obs_chunk[i])
    w_chunk_avg = jnp.mean(w_chunk)
    e_chunk_avg = jnp.mean(e_chunk * w_chunk) / w_chunk_avg
    elapsed = time.perf_counter() - t0
    dt_per_block = (time.perf_counter() - t_mark) / float(n)
    t_mark = time.perf_counter()
    stats = blocking_analysis_ratio(
        jnp.asarray(block_e_s), jnp.asarray(block_w_s), print_q=False
    )
    mu = stats["mu"]
    se = stats["se_star"]
    nodes = int(state.node_encounters)
    print(
        f"[blk {start + n:4d}/{params.n_blocks}]  "
        f"{mu:14.10f}  "
        f"{(f'{se:10.3e}' if se is not None else ' ' * 10)}  "
        f"{float(e_chunk_avg):16.10f}  "
        f"{float(w_chunk_avg):12.6e}  "
        f"{nodes:10d}  "
        f"{dt_per_block:9.3f}  "
        f"{elapsed:8.1f}"
    )
    if se is not None and se <= target_error and target_error > 0.0:
        print(f"\nTarget error {target_error:.3e} reached at block {start + n}.")
        break
block_e_s = jnp.asarray(block_e_s)
block_w_s = jnp.asarray(block_w_s)
block_obs_s = {
    name: (jnp.concatenate(block_obs_s[name], axis=0) if len(block_obs_s[name]) > 0 else None)
    for name in observable_names
}

data_clean, keep_mask = reject_outliers(jnp.column_stack((block_e_s, block_w_s)), obs=0)
print(f"\nRejected {block_e_s.shape[0] - data_clean.shape[0]} outlier blocks.")
block_e_s = jnp.asarray(data_clean[:, 0])
block_w_s = jnp.asarray(data_clean[:, 1])
keep_mask = jnp.asarray(keep_mask)
block_obs_s = {
    name: (arr[keep_mask] if arr is not None else None) for name, arr in block_obs_s.items()
}
print("\nFinal blocking analysis:")
stats = blocking_analysis_ratio(block_e_s, block_w_s, print_q=True)
mean, err = stats["mu"], stats["se_star"]

# block_e_all = jnp.concatenate([block_e_eq, block_e_s])
# block_w_all = jnp.concatenate([block_w_eq, block_w_s])
# block_obs_all: dict[str, jax.Array] = {}
# for name in observable_names:
#     arr_eq = block_obs_eq[name]
#     arr_s = block_obs_s[name]
#     if arr_eq is None and arr_s is None:
#         block_obs_all[name] = jnp.zeros((0,))
#         continue
#     if arr_eq is None:
#         arr_eq = arr_s[:0]
#     if arr_s is None:
#         arr_s = arr_eq[:0]
#     block_obs_all[name] = jnp.concatenate([arr_eq, arr_s], axis=0)

# obs_means: dict[str, jax.Array] = {}
# obs_stderrs: dict[str, jax.Array] = {}
# b_star = stats.get("B_star")
# for name in observable_names:
#     arr = block_obs_s[name]
#     if arr is None:
#         obs_means[name] = jnp.zeros((0,))
#         obs_stderrs[name] = jnp.zeros((0,))
#         continue
#     obs_means[name] = _weighted_block_mean(arr, block_w_s)
#     if b_star is not None and b_star >= 1:
#         import numpy as np

#         num, denom = rebin_observable(np.asarray(arr), np.asarray(block_w_s), b_star)
#         if num.shape[0] >= 2:
#             _, se = jackknife_ratios(num, denom)
#             obs_stderrs[name] = jnp.asarray(se)
#         else:
#             obs_stderrs[name] = jnp.full(arr.shape[1:], jnp.nan)
#     else:
#         obs_stderrs[name] = jnp.full(arr.shape[1:], jnp.nan)


Equilibration:

        block           E_blk             W        nodes      t[s]
[eql    0/100]   -1.0564298822  2.000000e+02           0       0.0
[eql   20/100]   -1.0922313954  2.001568e+02           0       1.1
[eql   40/100]   -1.0960047250  2.000169e+02           0       1.9
[eql   60/100]   -1.0921494467  1.999850e+02           0       2.1
[eql   80/100]   -1.0906510809  1.999956e+02           0       2.3
[eql  100/100]   -1.0896622800  1.999817e+02           0       2.6

Sampling:

        block           E_avg       E_err         E_block             W         nodes    dt[s/bl]     t[s]
[blk   20/200]   -1.0912183469   1.373e-03     -1.0912183469  1.999901e+02           0      0.149       3.0
[blk   40/200]   -1.0935370008   1.330e-03     -1.0958555787  1.999966e+02           0      0.017       3.3
[blk   60/200]   -1.0931479234   1.167e-03     -1.0923695949  1.999487e+02           0      0.016       3.6
[blk   80/200]   -1.0936076629   1.249e-03     -1.0949866682  2.000094e

In [ ]:
from typing import Any, Callable, NamedTuple, Protocol

import jax
import jax.numpy as jnp
from jax import lax, tree_util

from ad_afqmc_prototype import walkers as wk
from ad_afqmc_prototype.core.levels import LevelPack
from ad_afqmc_prototype.core.ops import MeasOps, TrialOps, k_energy
from ad_afqmc_prototype.core.system import System
from ad_afqmc_prototype.walkers import SrFn
from ad_afqmc_prototype.prop.types import PropOps, PropState, QmcParams, QmcParamsFp
from ad_afqmc_prototype.prop.blocks import BlockFn
from ad_afqmc_prototype.stat_utils import blocking_analysis_ratio, jackknife_ratios, rebin_observable, reject_outliers
from ad_afqmc_prototype.walkers import stochastic_reconfiguration

from jax import lax
from jax.sharding import Mesh, NamedSharding
from jax.sharding import PartitionSpec as P

from ad_afqmc_prototype.driver import QmcResult, make_run_blocks
from functools import partial
import time
print = partial(print, flush=True)

def run_pt2qmc(
    *,
    sys: System,
    params: QmcParams,
    ham_data: Any,
    guide_data: Any,
    meas_ops: MeasOps,
    guide_ops: TrialOps,
    prop_ops: PropOps,
    block_fn: BlockFn,
    state: PropState | None = None,
    meas_ctx: Any | None = None,
    target_error: float | None = None,
    mesh: Mesh | None = None,
    observable_names: tuple[str, ...] = (),
) -> QmcResult:
    """
    equilibration blocks with Guiding wavefunction (currently HF)
    then sampling blocks with pt2CCSD trial

    Returns:
      pt2QmcResult with energy statistics plus block-level observable estimates.
    """
    for name in observable_names:
        meas_ops.require_observable(name)

    # build ctx
    prop_ctx = prop_ops.build_prop_ctx(ham_data, guide_ops.get_rdm1(guide_data), params)
    if meas_ctx is None:
        meas_ctx = meas_ops.build_meas_ctx(ham_data, guide_data)
    if state is None:
        state = prop_ops.init_prop_state(
            sys=sys,
            ham_data=ham_data,
            trial_ops=trial_ops,
            trial_data=trial_data,
            meas_ops=meas_ops,
            params=params,
            mesh=mesh,
        )

    if mesh is None or mesh.size == 1:
        block_fn_sr = block_fn
    else:
        data_sh = NamedSharding(mesh, P("data"))
        sr_sharded = partial(stochastic_reconfiguration, data_sharding=data_sh)
        block_fn_sr = partial(block_fn, sr_fn=sr_sharded)

    run_guide_blocks = make_run_blocks(
        block_fn=block_fn_sr,
        sys=sys,
        params=params,
        trial_ops=guide_ops,
        meas_ops=meas_ops,
        prop_ops=prop_ops,
        observable_names=observable_names,
    )

    run_trial_blocks = make_run_pt2blocks(
        block_fn=block_fn_sr,
        sys=sys,
        params=params,
        trial_ops=guide_ops,
        meas_ops=meas_ops,
        prop_ops=prop_ops,
        observable_names=observable_names,
    )

    t0 = time.perf_counter()
    t_mark = t0

    print_every = params.n_eql_blocks // 5 if params.n_eql_blocks >= 5 else 0
    block_e_eq = []
    block_w_eq = []
    block_obs_eq = {name: [] for name in observable_names}
    block_e_eq.append(state.e_estimate)
    block_w_eq.append(jnp.sum(state.weights))
    print("\nEquilibration:\n")
    if print_every:
        print(
            f"{'':4s}"
            f"{'block':>9s}  "
            f"{'E_blk':>14s}  "
            f"{'W':>12s}   "
            f"{'nodes':>10s}  "
            f"{'t[s]':>8s}"
        )
    print(
        f"[eql {0:4d}/{params.n_eql_blocks}]  "
        f"{float(state.e_estimate):14.10f}  "
        f"{float(jnp.sum(state.weights)):12.6e}  "
        f"{int(state.node_encounters):10d}  "
        f"{0.0:8.1f}"
    )
    chunk = print_every if print_every > 0 else 1
    for start in range(0, params.n_eql_blocks, chunk):
        n = min(chunk, params.n_eql_blocks - start)
        state, scalars_chunk, obs_chunk = run_eql_blocks(
            state,
            ham_data=ham_data,
            trial_data=guide_data,
            meas_ctx=meas_ctx,
            prop_ctx=prop_ctx,
            n_blocks=n,
        )
        block_e_eq.extend(scalars_chunk["energy"].tolist())
        block_w_eq.extend(scalars_chunk["weight"].tolist())
        for i, name in enumerate(observable_names):
            block_obs_eq[name].append(obs_chunk[i])
        e_chunk = scalars_chunk["energy"]
        w_chunk = scalars_chunk["weight"]
        w_chunk_avg = jnp.mean(w_chunk)
        e_chunk_avg = jnp.mean(e_chunk * w_chunk) / w_chunk_avg
        elapsed = time.perf_counter() - t0
        print(
            f"[eql {start + n:4d}/{params.n_eql_blocks}]  "
            f"{float(e_chunk_avg):14.10f}  "
            f"{float(w_chunk_avg):12.6e}  "
            f"{int(state.node_encounters):10d}  "
            f"{elapsed:8.1f}"
        )
    block_e_eq = jnp.asarray(block_e_eq)
    block_w_eq = jnp.asarray(block_w_eq)
    block_obs_eq = {
        name: (jnp.concatenate(block_obs_eq[name], axis=0) if len(block_obs_eq[name]) > 0 else None)
        for name in observable_names
    }

    # sampling
    print("\nSampling:\n")
    if target_error is None:
        target_error = 0.0
    print_every = params.n_blocks // 10 if params.n_blocks >= 10 else 0
    block_e_s = []
    block_w_s = []
    block_obs_s = {name: [] for name in observable_names}
    if print_every:
        print(
            f"{'':4s}{'block':>9s}  {'E_avg':>14s}  {'E_err':>10s}  {'E_block':>14s}  "
            f"{'W':>12s}    {'nodes':>10s}  {'dt[s/bl]':>10s}  {'t[s]':>7s}"
        )

    chunk = print_every if print_every > 0 else 1
    for start in range(0, params.n_blocks, chunk):
        n = min(chunk, params.n_blocks - start)
        state, scalars_chunk, obs_chunk = run_smp_blocks(
            state,
            ham_data=ham_data,
            trial_data=trial_data,
            meas_ctx=meas_ctx,
            prop_ctx=prop_ctx,
            n_blocks=n,
        )
        e_chunk = scalars_chunk["energy"]
        w_chunk = scalars_chunk["weight"]
        block_e_s.extend(e_chunk.tolist())
        block_w_s.extend(w_chunk.tolist())
        for i, name in enumerate(observable_names):
            block_obs_s[name].append(obs_chunk[i])
        w_chunk_avg = jnp.mean(w_chunk)
        e_chunk_avg = jnp.mean(e_chunk * w_chunk) / w_chunk_avg
        elapsed = time.perf_counter() - t0
        dt_per_block = (time.perf_counter() - t_mark) / float(n)
        t_mark = time.perf_counter()
        stats = blocking_analysis_ratio(
            jnp.asarray(block_e_s), jnp.asarray(block_w_s), print_q=False
        )
        mu = stats["mu"]
        se = stats["se_star"]
        nodes = int(state.node_encounters)
        print(
            f"[blk {start + n:4d}/{params.n_blocks}]  "
            f"{mu:14.10f}  "
            f"{(f'{se:10.3e}' if se is not None else ' ' * 10)}  "
            f"{float(e_chunk_avg):16.10f}  "
            f"{float(w_chunk_avg):12.6e}  "
            f"{nodes:10d}  "
            f"{dt_per_block:9.3f}  "
            f"{elapsed:8.1f}"
        )
        if se is not None and se <= target_error and target_error > 0.0:
            print(f"\nTarget error {target_error:.3e} reached at block {start + n}.")
            break
    block_e_s = jnp.asarray(block_e_s)
    block_w_s = jnp.asarray(block_w_s)
    block_obs_s = {
        name: (jnp.concatenate(block_obs_s[name], axis=0) if len(block_obs_s[name]) > 0 else None)
        for name in observable_names
    }

    data_clean, keep_mask = reject_outliers(jnp.column_stack((block_e_s, block_w_s)), obs=0)
    print(f"\nRejected {block_e_s.shape[0] - data_clean.shape[0]} outlier blocks.")
    block_e_s = jnp.asarray(data_clean[:, 0])
    block_w_s = jnp.asarray(data_clean[:, 1])
    keep_mask = jnp.asarray(keep_mask)
    block_obs_s = {
        name: (arr[keep_mask] if arr is not None else None) for name, arr in block_obs_s.items()
    }
    print("\nFinal blocking analysis:")
    stats = blocking_analysis_ratio(block_e_s, block_w_s, print_q=True)
    mean, err = stats["mu"], stats["se_star"]

    block_e_all = jnp.concatenate([block_e_eq, block_e_s])
    block_w_all = jnp.concatenate([block_w_eq, block_w_s])
    block_obs_all: dict[str, jax.Array] = {}
    for name in observable_names:
        arr_eq = block_obs_eq[name]
        arr_s = block_obs_s[name]
        if arr_eq is None and arr_s is None:
            block_obs_all[name] = jnp.zeros((0,))
            continue
        if arr_eq is None:
            arr_eq = arr_s[:0]
        if arr_s is None:
            arr_s = arr_eq[:0]
        block_obs_all[name] = jnp.concatenate([arr_eq, arr_s], axis=0)

    obs_means: dict[str, jax.Array] = {}
    obs_stderrs: dict[str, jax.Array] = {}
    b_star = stats.get("B_star")
    for name in observable_names:
        arr = block_obs_s[name]
        if arr is None:
            obs_means[name] = jnp.zeros((0,))
            obs_stderrs[name] = jnp.zeros((0,))
            continue
        obs_means[name] = _weighted_block_mean(arr, block_w_s)
        if b_star is not None and b_star >= 1:
            import numpy as np

            num, denom = rebin_observable(np.asarray(arr), np.asarray(block_w_s), b_star)
            if num.shape[0] >= 2:
                _, se = jackknife_ratios(num, denom)
                obs_stderrs[name] = jnp.asarray(se)
            else:
                obs_stderrs[name] = jnp.full(arr.shape[1:], jnp.nan)
        else:
            obs_stderrs[name] = jnp.full(arr.shape[1:], jnp.nan)

    return QmcResult(
        mean_energy=mean,
        stderr_energy=err,
        block_energies=block_e_all,
        block_weights=block_w_all,
        block_observables=block_obs_all,
        observable_means=obs_means,
        observable_stderrs=obs_stderrs,
    )

In [79]:
from ad_afqmc_prototype.core.system import System
from ad_afqmc_prototype.prop.afqmc import make_prop_ops, init_prop_state
from ad_afqmc_prototype.prop.types import QmcParams
from ad_afqmc_prototype.prop.blocks import block as qmc_block

sys_obj   = System(norb=norb, nelec=(nocc, nocc), walker_kind="restricted")
trial_ops = make_pt2ccsd_trial_ops(sys_obj)
meas_ops  = make_pt2ccsd_meas_ops(sys_obj, memory_mode="low")
prop_ops  = make_prop_ops(ham.basis, sys_obj.walker_kind)

params = QmcParams(
    dt            = 0.005,   # imaginary-time step
    n_walkers     = 50,      # walker population
    n_prop_steps  = 10,      # propagation steps per block
    n_blocks      = 200,     # sampling blocks
    n_eql_blocks  = 10,      # equilibration blocks (discarded)
    seed          = 17,
)

print(f"dt={params.dt}  n_walkers={params.n_walkers}  n_prop_steps={params.n_prop_steps}")
print(f"total imaginary time sampled: {params.n_blocks * params.n_prop_steps * params.dt:.2f} a.u.")

dt=0.005  n_walkers=50  n_prop_steps=10
total imaginary time sampled: 10.00 a.u.


In [55]:
trial_ops.get_rdm1

<function ad_afqmc_prototype.trial.rhf.get_rdm1(trial_data: 'RhfTrial') -> 'jax.Array'>

In [80]:
rdm1      = trial_ops.get_rdm1(trial)
prop_ctx  = prop_ops.build_prop_ctx(ham, rdm1, params)
meas_ctx_prop = meas_ops.build_meas_ctx(ham, trial)

state = init_prop_state(
    sys=sys_obj,
    ham_data=ham,
    trial_ops=trial_ops,
    trial_data=trial,
    meas_ops=meas_ops,
    params=params,
)

print(f"Initial walker batch shape: {state.walkers.shape}")
print(f"Initial weights (first 5):  {state.weights[:5]}")
print(f"Initial e_estimate: {float(state.e_estimate):.8f} Ha")
print(f"Initial mean |overlap|: {float(jnp.mean(jnp.abs(state.overlaps))):.6f}")

TypeError: energy_kernel_rw_rh() takes 3 positional arguments but 4 were given

In [ ]:
import time

print(f"Equilibrating for {params.n_eql_blocks} blocks ...")
t0 = time.time()

for i in range(params.n_eql_blocks):
    state, obs = qmc_block(
        state,
        sys=sys_obj,
        params=params,
        ham_data=ham,
        trial_data=trial,
        trial_ops=trial_ops,
        meas_ops=meas_ops,
        meas_ctx=meas_ctx_prop,
        prop_ops=prop_ops,
        prop_ctx=prop_ctx,
    )
    # if i % 10 == 0 or i == params.n_eql_blocks - 1:
    e   = float(obs.scalars["energy"])
    w   = float(obs.scalars["weight"])
    est = float(state.e_estimate)
    print(f"  eql {i:3d}: E={e:.8f}  total_w={w:.4f}  e_est={est:.8f}")

print(f"Equilibration done in {time.time()-t0:.1f}s")